# Segment nuclei using your trained StarDist-3D model

The models used in [Ferme et al 2024](https://www.biorxiv.org/content/10.1101/2024.11.12.623216v1) were trained on a training dataset having a voxel size ofnearly 0.24 $\mu$m in z and around 0.10-0.12 $\mu$m in xy, hence the ratio between the xy pixel and z step size is around 2.4-2.0

In [ ]:
from __future__ import print_function, unicode_literals, absolute_import, division
import os
from glob import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tifffile import imread, imsave, imwrite

Downscale = True
Extract_singleNucleusFeatures = True,
Perform_SkeletonAnalysis = False,
Perform_NeighbourStatistics = False,
Perform_LocalAveraging = True,


### Local functions
from PIL import Image
from PIL.TiffTags import TAGS
def get_spatial_calib_tiff(image):
    """ Gets the path/ title of a tiff file and returns the zyx pixel sizes in microns"""
    with Image.open(image) as img:
        meta_dict = {TAGS[key] : img.tag[key] for key in img.tag.keys()}

    # unit: micron
    z = [w[-4:] for w in meta_dict['ImageDescription'][0].split("\n") if w.startswith('spacing')]
    x = 1/ (meta_dict["XResolution"][0][0]/meta_dict["XResolution"][0][1])
    y = 1/ (meta_dict["YResolution"][0][0]/meta_dict["YResolution"][0][1])

    return float(z[0]), float("%.4f" % y), float("%.4f" % x)

def save_image(img,calib,outdir,savename):
    """Saves an image to imagej tiff hyperstack format. Adds spatial calibration (pixelsize)
    to meta-data. The axis order of the input img is important (the image is a numpy array 
    of shape (T,C,Z,Y,X) or (T,Z,Y,X))
    """
    # create outputdir
    if not os.path.exists(outdir):
        os.makedirs(outdir)

    fullsavename=os.path.join(outdir,savename)
    print("Image dimensions", img.ndim)
    if img.ndim==5: # TCZYX
        outimg=np.swapaxes(img,1,2) # -> TZCYX
        axes = "TZCYX"
    elif img.ndim ==4:
        outimg=np.swapaxes(img, 0, 1) # -> ZCYX
        axes = "ZCYX"
    elif img.ndim == 3:
        outimg = img
        axes = "ZYX"
    elif img.ndim > 5:
        outimg = img.squeeze()
        if outimg.ndim <= 5:
            save_image(outimg, calib, outdir, savename)
        else:
            print("The dimensions of the image are greater than 5, please correct them.")

    # for tifffile saving: help(tifffile) and https://forum.image.sc/t/python-copy-all-metadata-from-one-multipage-tif-to-another/26597/8
    tifffile.imsave(fullsavename,outimg, imagej=True, resolution=(1/calib[0], 1/calib[1]),
            metadata={'spacing': calib[2],
                    'axes': axes,
                    'unit': 'um'})

### 1. Downscale
The objects that you want to analyze need to be fully contained within the field of view of the model. Because of this, it might be necessary to reduce the size of your image. 

In [ ]:
from skimage import transform

### Path to the folder where your .tif files and to output folder
inputdir = '...'  ### args.pathname
outputdir = os.path.join(inputdir, "images")

### Parameters
downsample_xy = 2

### Read files and process them
files = sorted(glob.glob(inputdir+"/*.tif"))# [el for el in os.listdir("./") if el.endswith(".czi")]
print("Found {} files in input directory".format(len(files)))
for i in range(0,len(files)):
    print("loading image {} of {} ".format(i+1, len(files)))
    print(os.path.basename(files[i]))

    # read image
    img = imread(files[i])

    # get voxel size, i.e. original calibration
    calib_orig = get_spatial_calib_tiff(files[i])

    # crop and downscale image
    downscaled_stack = transform.downscale_local_mean(img, factors=(1,1,1,downsample_xy,downsample_xy),
                        cval=np.mean(img)).astype(np.uint16)
    # adjust calibration
    downscaled_calib = [calib_orig[0]*downsample_xy,calib_orig[1]*downsample_xy,calib_orig[2]]
    
    # Save downscaled image
    basename= os.path.basename(files[i]).rsplit(".")[0]
    savename= basename+"_preprocessed.tif"

    save_image(downscaled_stack, downscaled_calib, outputdir, savename)
    print("\nDONE SAVING.")


### 2. Run StarDist-3D predictions

2.1 Define the directories where to save your predictions and the size of the tiles

In [ ]:
import sys
from numpy.lib.function_base import copy
from tqdm import tqdm
from csbdeep.utils import Path, normalize
from csbdeep.io import save_tiff_imagej_compatible
from stardist import random_label_cmap
from stardist.models import StarDist3D
from stardist import export_imagej_rois

import tensorflow as tf

### Check that you can access the GPU
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
  try:
    for gpu in gpus:
      tf.config.experimental.set_memory_growth(gpu, True) #limited to 5700 Mb over 8000 Mb, need better approach
  except RuntimeError as e:
    print(e)

### -----------------------------------------------------------------------------------------------------------------------------------------------###
np.random.seed(6)
lbl_cmap = random_label_cmap()

### Path to folder containing the "images" folder
pathname = inputdir

### Define the path where the model you want to use is located
version = "" # name of the model
model_name = "model_"+version
model_dir = "C:/Users/lcferme/Desktop/stardist/models"
print("Model Directory ", model_dir, "\n")

### Create output folder for the results from the prediction
predictions_dir = pathname+"/results"+version
if not os.path.exists(predictions_dir):
    os.mkdir(predictions_dir)
print("Output folder for StarDist predictions: ",predictions_dir)

### Define tiles for prediction
Automatic_number_of_tiles = True
## If you get an Out of memory (OOM) error when using the "Automatic_number_of_tiles" option, disable it and manually input the values to be used to process your images.  Progressively increases these numbers until the OOM error disappear.
n_tiles_Z =  2
n_tiles_Y =  8
n_tiles_X =  8

if (Automatic_number_of_tiles):
  n_tiles = None

if not (Automatic_number_of_tiles):
  n_tiles = (n_tiles_Z, n_tiles_Y, n_tiles_X)


2.2 Run the model and save the predictions

In [ ]:
### Read .tif files
files = sorted(glob(pathname+"/images/*.tif"))
print('\nNumber of test dataset found in the folder: '+str(len(files)))
Z = list(map(imread,files))

### Normalize
print("Found {} files and they have {} shape".format(len(Z), Z[0].ndim))
n_channel = 1 if Z[0].ndim == 3 else Z[0].shape[1]
axis_norm = (0,1,2)   # normalize channels independently
# axis_norm = (0,1,2,3) # normalize channels jointly
if n_channel > 1:
    print("Number of channels =", n_channel,"\nNormalizing image channels %s." % ('jointly' if axis_norm is None or 2 in axis_norm else 'independently'))
    if Z[0].shape[:-1]!=n_channel:
      Z = [normalize(z[:, 0, :, :], 1,99.8) for z in Z] #usually the h2b channel is the first (0)
else:
  Z = [normalize(z, 1,99.8) for z in Z]

### Check that the shape of each file is correct
for z in Z:
  print("shape", z.shape, z.ndim)

### Collect files' basenames 
predictions_paths=[]
names = [os.path.basename(f) for f in sorted(glob(pathname+"/images/*.tif"))]
for n in names:
    n_dir = os.path.join(predictions_dir, n)
    predictions_paths.append(n_dir)

### Get StarDist-3D model
model = StarDist3D(None, name=model_name, basedir=model_dir)

### Run predictions
for i in range(len(Z)):
    print('\nPREDICT image size ', z.shape, n_tiles)
    z_res, y_res, x_res = get_spatial_calib_tiff(files[i])

    labels, polygons = model.predict_instances(Z[i],  n_tiles = (n_tiles_Z,n_tiles_Y,n_tiles_X))#, prob_thresh=0.2)#axes="ZYX"
    print("PREDICTION IS COMPLETED")
    
    ### Save the predicted mask in the result folder 
    labels = np.array(labels, dtype='uint16')
    imwrite(predictions_paths[i], labels, imagej=True, resolution=(1/x_res, 1/y_res), metadata={'spacing': z_res,'axes': 'ZYX','unit': 'um'}) #image for imagej is like xyz no?
    print("\nDONE SAVING.")
